In [28]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio

In [29]:
datapath = "data/"

df_history = pd.read_csv(f"{datapath}europe_yearly_full_release_long_format.csv")
df_prices = pd.read_csv(f"{datapath}european_wholesale_electricity_price_data_monthly.csv")

xls = pd.ExcelFile(f"{datapath}targets_download.xlsx")
print(f"found {len(xls.sheet_names)} sheets in targets file: {xls.sheet_names}")

found 4 sheets in targets file: ['raw_data_long', 'sources', 'capacity_target_wide', 'share_target_wide']


In [30]:
df_targets = pd.read_excel(f"{datapath}targets_download.xlsx", sheet_name='raw_data_long')

In [31]:
df_history.head()

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity demand,Demand,Demand,TWh,48.81,NaN,NaN
1,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity demand,Demand per capita,Demand per capita,MWh,6.36,NaN,NaN
2,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,%,66.25,NaN,NaN
3,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Coal,%,12.56,NaN,NaN
4,Austria,AUT,1990,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,%,33.75,NaN,NaN


In [32]:
datasets = [df_history, df_targets, df_prices]

for df in datasets:
    print(f"Shape of df: {df.shape}")
    print(f"Available columns in df: {df.columns}")

Shape of df: (69310, 18)
Available columns in df: Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region',
       'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory',
       'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change'],
      dtype='str')
Shape of df: (2143, 8)
Available columns in df: Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')
Shape of df: (3801, 4)
Available columns in df: Index(['Country', 'ISO3 Code', 'Date', 'Price (EUR/MWhe)'], dtype='str')


In [33]:
df_history_cleaned = df_history[
    (df_history['Year'] == 2023) & 
    (df_history['Category'] == 'Electricity generation') &
    (df_history['Unit'] == 'TWh')
].copy()

df_history_cleaned[df_history_cleaned["Area"] == "Austria"]

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
1758,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,TWh,60.46,10.04,19.91
1759,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Coal,TWh,0.00,0.00,NaN
1760,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,TWh,10.82,-3.25,-23.10
1761,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, bioenergy and other renewables",TWh,45.23,5.85,14.86
1762,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Renewables,TWh,60.46,10.04,19.91
1763,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Wind,TWh,8.34,1.09,15.03
1764,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Wind and solar,TWh,15.23,4.19,37.95
1775,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Fuel,Bioenergy,TWh,4.56,-0.19,-4.00
1776,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Fuel,Gas,TWh,7.50,-3.35,-30.88
1777,Austria,AUT,2023,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Fuel,Hard coal,TWh,0.00,0.00,NaN


In [34]:
df_targets.columns

Index(['COUNTRY_CODE', 'TARGET_YEAR', 'FUEL_CATEGORY', 'METRIC', 'VALUE',
       'SOURCE_ID', 'NUMBER_OF_UPDATES', 'COLLECTION_YEAR'],
      dtype='str')

In [35]:
# identify available target metrics and find the one that matches the history data
print(f"Metrics available in df_targets: {df_targets["METRIC"].unique()}")

# gonna use 'generation_twh'

df_targets_cleaned = df_targets[
    (df_targets["METRIC"] == "generation_twh") &
    (df_targets["TARGET_YEAR"] == 2030)
].copy()

Metrics available in df_targets: <StringArray>
[       'capacity_total_gw', 'share_of_consumption_pct',
           'generation_twh',  'share_of_generation_pct',
   'capacity_additional_mw',               'demand_twh',
               'import_twh']
Length: 7, dtype: str


In [36]:
# pick columns and rename to make the data comparable

df_history_cleaned = df_history_cleaned[["ISO 3 code", "Variable", "Value"]]
df_targets_cleaned = df_targets_cleaned[["COUNTRY_CODE", "FUEL_CATEGORY", "VALUE"]]

df_history_cleaned = df_history_cleaned.rename(columns = {"ISO 3 code": "iso", "Variable": "source", "Value": "value"})
df_targets_cleaned = df_targets_cleaned.rename(columns = {"COUNTRY_CODE": "iso", "FUEL_CATEGORY": "source", "VALUE": "value"})

In [37]:
df_history_cleaned

,iso,source,value
1758,AUT,Clean,60.46
1759,AUT,Coal,0.00
1760,AUT,Fossil,10.82
1761,AUT,"Hydro, bioenergy and other renewables",45.23
1762,AUT,Renewables,60.46
...,...,...,...
69187,GBR,Offshore wind,49.65
69188,GBR,Onshore wind,32.50
69189,GBR,Other fossil,10.82
69190,GBR,Solar,13.88


In [38]:
df_history_cleaned["source"].unique()

<StringArray>
[                                'Clean',
                                  'Coal',
                                'Fossil',
 'Hydro, bioenergy and other renewables',
                            'Renewables',
                                  'Wind',
                        'Wind and solar',
                             'Bioenergy',
                                   'Gas',
                             'Hard coal',
                                 'Hydro',
                               'Lignite',
                               'Nuclear',
                          'Onshore wind',
                          'Other fossil',
                      'Other renewables',
                                 'Solar',
                      'Total generation',
                         'Offshore wind']
Length: 19, dtype: str

In [39]:
df_targets_cleaned["source"].unique()

<StringArray>
[                          'Solar',                            'Wind',
                       'Bioenergy',                           'Hydro',
                'Other Renewables',                            'Coal',
                             'Gas',                         'Nuclear',
                    'Other Fossil',                      'Renewables',
                    'Onshore Wind',                   'Offshore Wind',
                          'Fossil', 'Hydro, bio and other renewables',
                          'Demand',            'Non-hydro renewables',
              'Rest of renewables',                         'Storage',
                    'Pumped hydro',                         'Battery',
                         'Imports']
Length: 21, dtype: str

In [40]:
# sort the "source" column in fossil and renewables
# note on columns not considered:
# battery/storage/pumped hydro: no info about the source, 

RENEWABLES = {
    "solar", "wind", "onshore wind", "offshore wind", "wind and solar",
    "bioenergy", "hydro", "other renewables", "renewables",
    "hydro, bio and other renewables", "hydro, bioenergy and other renewables",
    "non-hydro renewables", "rest of renewables",
}

RENEWABLES_EVALUATED = {
    "solar",
}

FOSSIL = {
    "coal", "hard coal", "lignite", "gas", "other fossil", "fossil",
}

def categorize(value):
    if pd.isna(value):
        return pd.NA
    v = str(value).strip().lower()
    if v in RENEWABLES_EVALUATED:
        return "Renewables"
    if v in FOSSIL:
        return "Fossil"
    return "Other"

df_targets_cleaned["category"] = df_targets_cleaned["source"].map(categorize)
df_history_cleaned["category"] = df_history_cleaned["source"].map(categorize)

print(f"Unique values in df_targets_cleaned: {df_targets_cleaned["category"].unique()}")
print(f"Unique values in df_history_cleaned: {df_history_cleaned["category"].unique()}")

df_targets_cleaned_grouped = df_targets_cleaned.groupby(by=["category", "iso"]).sum()

df_history_cleaned_grouped = df_history_cleaned.groupby(by=["category", "iso"]).sum()

Unique values in df_targets_cleaned: <StringArray>
['Renewables', 'Other', 'Fossil']
Length: 3, dtype: str
Unique values in df_history_cleaned: <StringArray>
['Other', 'Fossil', 'Renewables']
Length: 3, dtype: str


In [41]:
df_gap = pd.merge(df_history_cleaned_grouped, df_targets_cleaned_grouped, on=["iso", "category"], how="inner", suffixes=["_history", "_target"])

df_gap["gap_twh"] = df_gap["value_target"] - df_gap["value_history"]
df_gap["pct_missing"] = (df_gap["gap_twh"] / df_gap["value_target"]) * 100

In [42]:
df_gap

,,source_history,value_history,source_target,value_target,gap_twh,pct_missing
iso,category,,,,,,
AUT,Fossil,CoalFossilGasHard coalLigniteOther fossil,21.64,GasFossilGasFossilGas,19.000,-2.640,-13.894737
BEL,Fossil,CoalFossilGasHard coalLigniteOther fossil,43.48,Gas,24.650,-18.830,-76.389452
BGR,Fossil,CoalFossilGasHard coalLigniteOther fossil,39.47,FossilFossil,10.474,-28.996,-276.837884
CHE,Fossil,FossilOther fossil,2.08,Other Fossil,1.600,-0.480,-30.000000
CYP,Fossil,CoalFossilGasHard coalLigniteOther fossil,8.48,CoalGasCoalGas,7.564,-0.916,-12.109995
...,...,...,...,...,...,...,...
ROU,Renewables,Solar,2.23,SolarSolar,21.700,19.470,89.723502
SVK,Renewables,Solar,0.61,Solar,1.470,0.860,58.503401
SVN,Renewables,Solar,0.98,SolarSolar,3.757,2.777,73.915358


In [43]:
# Create an Interactive Bar Chart
df_gap = df_gap.reset_index()
df_graph =  df_gap[df_gap["category"] == 'Renewables'].sort_values(by='value_target', ascending=True)
df_graph = df_graph.rename(columns={"iso": "country", "value_history": "current_twh"})

fig = px.bar(
    df_graph, 
    y='country', 
    x=['current_twh', 'gap_twh'], 
    title='<b>The Solar Reality Gap (2023 vs 2030 Target)</b><br><i>Dark Blue = Current Generation, Light Blue = Remaining Gap to Target</i>',
    labels={'value': 'Generation (TWh)', 'country': 'Country', 'variable': 'Metric'},
    orientation='h',  # Horizontal bars are easier to read for country names
    height=800,       # Tall chart to fit all countries
    color_discrete_map={'current_twh': '#1f77b4', 'gap_twh': '#aec7e8'} # Dark vs Light Blue
)

# Customizing the Layout for a Professional Look
fig.update_layout(
    xaxis_title="Electricity Generation (TWh)",
    legend_title_text="",
    showlegend=True,
    template="plotly_white",
    hovermode="y unified" # Shows both values when hovering over a country
)

# Fix the legend names
new_names = {'current_twh': 'Current (2023)', 'gap_twh': 'Remaining Gap'}
fig.for_each_trace(lambda t: t.update(name = new_names[t.name]))

fig.show()

## Prices

In [44]:
df_prices

,Country,ISO3 Code,Date,Price (EUR/MWhe)
0,Austria,AUT,2015-01-01,29.94
1,Belgium,BEL,2015-01-01,42.33
2,Czechia,CZE,2015-01-01,29.47
3,Denmark,DNK,2015-01-01,27.12
4,Estonia,EST,2015-01-01,33.84
...,...,...,...,...
3796,Slovakia,SVK,2026-02-01,122.36
3797,Slovenia,SVN,2026-02-01,124.98
3798,Spain,ESP,2026-02-01,20.37
3799,Sweden,SWE,2026-02-01,102.59


In [45]:
df_prices["Country"].unique()

<StringArray>
[        'Austria',         'Belgium',         'Czechia',         'Denmark',
         'Estonia',         'Finland',          'France',         'Germany',
          'Greece',         'Hungary',           'Italy',          'Latvia',
       'Lithuania',      'Luxembourg',     'Netherlands',          'Norway',
          'Poland',        'Portugal',         'Romania',        'Slovakia',
        'Slovenia',           'Spain',          'Sweden',     'Switzerland',
  'United Kingdom',        'Bulgaria',          'Serbia',         'Croatia',
         'Ireland',      'Montenegro', 'North Macedonia']
Length: 31, dtype: str

In [46]:
selected_countries = ["Germany", "Norway", "Austria", "Spain", "Italy"]
df_prices_filtered = df_prices[df_prices["Country"].isin(selected_countries)].copy()

df_prices_filtered["Date"] = pd.to_datetime(df_prices_filtered["Date"])
df_prices_filtered = df_prices_filtered.sort_values(by="Date")

df_prices_filtered

,Country,ISO3 Code,Date,Price (EUR/MWhe)
0,Austria,AUT,2015-01-01,29.94
7,Germany,DEU,2015-01-01,29.94
10,Italy,ITA,2015-01-01,50.84
15,Norway,NOR,2015-01-01,29.19
21,Spain,ESP,2015-01-01,51.48
...,...,...,...,...
3791,Norway,NOR,2026-02-01,103.70
3773,Austria,AUT,2026-02-01,126.48
3782,Germany,DEU,2026-02-01,104.11
3786,Italy,ITA,2026-02-01,130.38


In [47]:
# --- 4.2 Create Interactive Line Chart ---
fig = px.line(
    df_prices_filtered, 
    x='Date', 
    y='Price (EUR/MWhe)', 
    color='Country',
    title='<b>The Energy Crisis: Wholesale Electricity Prices (2015-2023)</b><br><i>Notice the divergence during the 2022 shock</i>',
    template='plotly_white',
    height=600
)

# --- 4.3 Add Storytelling Elements ---
# Highlight the 2022 Crisis Year
fig.add_vrect(
    x0="2022-01-01", x1="2022-12-31", 
    fillcolor="red", opacity=0.1, 
    layer="below", line_width=0,
    annotation_text="2022 Energy Crisis", annotation_position="top left"
)

# Customize Layout
fig.update_layout(
    xaxis_title="",
    yaxis_title="Price (EUR/MWh)",
    legend_title_text="Country",
    hovermode="x unified" # Shows all country prices for that month in one tooltip
)

fig.show()

### Correlation Analysis between Renewables rates and Wholesale Energy  in 2022

In [70]:
# calculate mean wholesale energy prices per year and country:
keep_cols = ["Country", "Price (EUR/MWhe)"]
df_prices_filtered.index = pd.to_datetime(df_prices_filtered["Date"])
df_prices_filtered_yearly = df_prices_filtered.groupby([df_prices_filtered[keep_cols].index.year, "Country"])["Price (EUR/MWhe)"].mean().reset_index()

print(f"Prices filtered yearly dataframe: \n {df_prices_filtered_yearly}")

Prices filtered yearly dataframe: 
     Date  Country  Price (EUR/MWhe)
0   2015  Austria         31.764167
1   2015  Germany         31.764167
2   2015    Italy         52.305000
3   2015   Norway         20.225833
4   2015    Spain         50.263333
5   2016  Austria         28.956667
6   2016  Germany         28.956667
7   2016    Italy         42.655000
8   2016   Norway         26.131667
9   2016    Spain         39.613333
10  2017  Austria         34.419167
11  2017  Germany         34.638333
12  2017    Italy         53.912500
13  2017   Norway         28.598333
14  2017    Spain         52.210000
15  2018  Austria         40.891667
16  2018  Germany         43.463333
17  2018    Italy         61.264167
18  2018   Norway         43.540000
19  2018    Spain         57.295833
20  2019  Austria         40.137500
21  2019  Germany         37.814167
22  2019    Italy         52.348333
23  2019   Norway         39.015000
24  2019    Spain         47.705000
25  2020  Austria         33

In [67]:
# calculate the renewables' ratio:
# sort the "source" column in fossil and renewables
# note on columns not considered:
# battery/storage/pumped hydro: no info about the source, 

# all categories classified as renewables:
RENEWABLES = {
    "solar", "wind", "onshore wind", "offshore wind", "wind and solar",
    "bioenergy", "hydro", "other renewables", "renewables",
    "hydro, bio and other renewables", "hydro, bioenergy and other renewables",
    "non-hydro renewables", "rest of renewables",
}

# those i  will evaluate:
RENEWABLES_EVALUATED = {
    "solar", "wind", "wind and solar",
}

RENEWABLES_EVALUATED = RENEWABLES

FOSSIL = {
    "coal", "hard coal", "lignite", "gas", "other fossil", "fossil",
}

def categorize(value):
    if pd.isna(value):
        return pd.NA
    v = str(value).strip().lower()
    if v in RENEWABLES_EVALUATED:
        return "Renewables"
    if v in FOSSIL:
        return "Fossil"
    return "Other"

df_targets_cleaned["category"] = df_targets_cleaned["source"].map(categorize)
df_history_cleaned["category"] = df_history_cleaned["source"].map(categorize)

print(f"Unique values in df_targets_cleaned: {df_targets_cleaned["category"].unique()}")
print(f"Unique values in df_history_cleaned: {df_history_cleaned["category"].unique()}")

keep_cols = ["iso", "category", "value"]

df_targets_cleaned_grouped = df_targets_cleaned[keep_cols].groupby(by=["category", "iso"]).sum().reset_index()
df_history_cleaned_grouped = df_history_cleaned[keep_cols].groupby(by=["category", "iso"]).sum().reset_index()

Unique values in df_targets_cleaned: <StringArray>
['Renewables', 'Fossil', 'Other']
Length: 3, dtype: str
Unique values in df_history_cleaned: <StringArray>
['Other', 'Fossil', 'Renewables']
Length: 3, dtype: str


In [68]:
df_history_cleaned_grouped

print(f"Note: value is in TWh of electricity generated! \n\n Dataframe: \n {df_history_cleaned_grouped}")

shares = df_history_cleaned_grouped.pivot_table(index="iso", columns="category", values="value", aggfunc="sum", fill_value=0)
shares["renewable_ratio"] = shares["Renewables"] / shares.sum(axis=1)

shares

Note: value is in TWh of electricity generated! 

 Dataframe: 
        category  iso   value
0        Fossil  AUT   21.64
1        Fossil  BEL   43.48
2        Fossil  BGR   39.47
3        Fossil  BIH   30.58
4        Fossil  CHE    2.08
..          ...  ...     ...
103  Renewables  SVK   20.13
104  Renewables  SVN   18.91
105  Renewables  SWE  379.36
106  Renewables  TUR  436.90
107  Renewables  XKX    1.76

[108 rows x 3 columns]


category,Fossil,Other,Renewables,renewable_ratio
iso,,,,
AUT,21.64,131.74,189.72,0.552958
BEL,43.48,175.49,97.90,0.308960
BGR,39.47,82.51,31.85,0.207047
BIH,30.58,23.48,20.09,0.270937
CHE,2.08,158.24,126.35,0.440751
CYP,8.48,6.38,3.40,0.186199
CZE,98.44,147.78,34.16,0.121835
DEU,569.58,792.20,959.72,0.413405
DNK,11.75,63.41,107.71,0.588998


In [ ]:
# now create scatter plot with the renewables' rate and electricity prices

# NOTES:
# make country info comparable
# take renewables ratio from 2022 also (currently 2023 is selected -> redo filtering from the top)